<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/Hash_trainer_parallel_sparse_optimized_compile(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Ball — Local, Slot-Conditioned Hash Search

This version keeps the machine **single-stage**: each ball still performs exactly one
hash lookup and then reads the emitted string from the deflector table.

The training hash is changed to make generalization possible without layering:

- **Locality:** a ball's address depends only on a configurable neighborhood around it.
- **Relative-position sharing:** the same `(relative offset, token)` feature reuses one
  hash coefficient at every absolute position.
- **Slot separation:** each ball position gets its own bucket region, so a digit emitted
  by one output slot cannot collide with a different slot's meaning.
- **Reuse-aware search:** primary class-purity/CE remains the objective; ties prefer using
  the same class/bucket pair repeatedly instead of creating fresh buckets.
- **Multi-start training:** several independent discrete searches are tried and the best
  training solution is kept.

The physical simulator is unchanged in spirit: `travel -> one lookup -> walk`.
No input-reading layers are added.


In [ ]:
import random

In [ ]:
import sys
import pandas as pd
from transformers import AutoTokenizer
from collections import Counter

# Load the tokenizer
tok = AutoTokenizer.from_pretrained("gpt2")

tiny_cases = []
val_cases = []

# Load the stories 40k dataset
print("Loading stories 40k dataset...")
df = pd.read_csv('/content/tiny_stories_next_40k.csv')
print(f"Dataset has {len(df)} total rows. We will use the first 200 rows.")

# Select the first 200 stories
df = df.head(100)

# Determine the column containing the story text
story_col = 'story' if 'story' in df.columns else ('text' if 'text' in df.columns else df.columns[0])
print(f"Using column: '{story_col}' for extracting stories.")

# Process and extract sliding window prompts and single next tokens
processed_stories = []
window_size = 30

for idx, row in df.iterrows():
    text = str(row[story_col]).strip() if pd.notna(row[story_col]) else ""
    if not text or text.lower() == "nan":
        continue

    tokens = tok.encode(text)
    # Slide a window across the entire story
    for i in range(len(tokens) - window_size):
        prompt_tokens = tokens[i : i + window_size]
        next_token = tokens[i + window_size]

        prompt = tok.decode(prompt_tokens).strip()
        output = tok.decode([next_token]).strip()

        if prompt and output:
            processed_stories.append((prompt, output))

# Split into train (approx 90%) and validation (approx 10%)
split_idx = int(len(processed_stories) * 0.9)
train_stories = processed_stories[:split_idx]
val_stories = processed_stories[split_idx:]

# Deduplicate strictly by prompt string
def deduplicate_by_prompt(cases):
    seen = set()
    deduped = []
    for prompt, target in cases:
        if prompt not in seen:
            seen.add(prompt)
            deduped.append((prompt, target))
    return deduped

tiny_cases = deduplicate_by_prompt(train_stories)
val_cases = deduplicate_by_prompt(val_stories)

print(f"Loaded {len(tiny_cases)} unique sliding window train cases into 'tiny_cases'")
print(f"Loaded {len(val_cases)} unique sliding window validation cases into 'val_cases'")

# Count duplicate prompts
prompts = [item[0] for item in tiny_cases]
prompt_counts = Counter(prompts)

unique_prompts = len(prompt_counts)
duplicate_prompts = sum(1 for count in prompt_counts.values() if count > 1)

print(f"Number of unique inputs (prompts): {unique_prompts}")
print(f"Number of inputs that have duplicates: {duplicate_prompts}")

Loading stories 40k dataset...
Dataset has 40000 total rows. We will use the first 200 rows.
Using column: 'text' for extracting stories.
Loaded 20333 unique sliding window train cases into 'tiny_cases'
Loaded 2260 unique sliding window validation cases into 'val_cases'
Number of unique inputs (prompts): 20333
Number of inputs that have duplicates: 0


In [ ]:
from transformers import AutoTokenizer

# Collect all unique token IDs across the prompts and stories
unique_token_ids = set()
for prompt, story in tiny_cases:
    unique_token_ids.update(tok.encode(prompt))
    unique_token_ids.update(tok.encode(story))

# Decode each token to show the exact list of string representations
token_list = [tok.decode([tid]) for tid in sorted(unique_token_ids)]

print(f"Total unique subword tokens: {len(token_list)}")
print("Token list:", token_list[:10])

Total unique subword tokens: 3568
Token list: ['!', '"', "'", ',', '-', '.', '3', ':', ';', '?']


In [ ]:
import random, time, math, difflib
import numpy as np

CFG = dict(
    local_D=1000,             # buckets per ball/output slot
    hash_radius=10,       # local neighborhood radius in token positions
    direction_range=1,     # learned feature offsets are searched in [-V, V]
    bounce=10,
    cap=5,
    n_cases=5000, train_share=0.8, seed=66,
    loss='ce',              # 'hard' or 'ce'
    reuse_tiebreak=0.01,    # secondary preference when primary loss ties
    restarts=1,              # independent discrete initializations
    rounds=1,                # >=3 so free resegmentation can iterate
    sweeps=1,
    ntry=12,
    free_reseg=True,         # option 1: resegment into ANY substring, not just known classes
    copy_split=True,         # option 2: content-anchored initial split (copy digit tokens, cut at word edges)
    sparse_mode='auto',       # candidate scoring: 'auto' (cheaper of sparse/dense per move), 'always', 'never'
)
random.seed(CFG['seed']); np.random.seed(CFG['seed'])


## Data: tokenizer and arithmetic generalization test

The training task is deliberately structured: `1+i=` -> `i+1`.
The train/test split is by **exact context**, while every test token is drawn from the
same small alphabet. This isolates whether the hash learns reusable structure.


In [ ]:
def TokenizeInput(input, alphabet):
    """Greedy non-overlapping tokenization: earliest start, then longest, then alphabet order."""
    input_str = str(input)
    matches = []
    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue
        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break
            matches.append((index, -len(item_str), order, item))
            start_pos = index + 1
    matches.sort(key=lambda m: (m[0], m[1], m[2]))
    out, last_end = [], 0
    for index, neg_len, _, item in matches:
        if index >= last_end:
            out.append(item)
            last_end = index - neg_len
    return out

# Extract the unique alphabet characters present across all TinyStories inputs
alphabet = token_list #sorted(list(set(char for inp, _ in tiny_cases for char in inp)))

output = token_list #sorted(set(ch for _, t in tiny_cases for ch in t))
max_input_len = 52 #max(len(TokenizeInput(inp, alphabet)) for inp, _ in tiny_cases)

print(
    f"alphabet={len(alphabet)}  output symbols={len(output)}  "
    f"train={len(tiny_cases)}  max_input_len={max_input_len}"
)
print("train example:", tiny_cases[:1])

alphabet=3568  output symbols=3568  train=20333  max_input_len=52
train example: [('Once upon a time there was a little boy. He was three years old and loved to eat. One day, he saw a spicy snack. He', 'wanted')]


## The single-stage brain

The hash is now a **local, slot-conditioned feature hash**:

```
h_i = slot(i) + [
        s[token_i]
        + Σ_{δ=-R..R, δ≠0} d[δ, token_{i+δ}]
      ] mod local_D
```

Only tokens inside radius `R` affect ball `i`.  The same relative-position feature
`(δ, token)` is shared across the whole dataset, while the bucket range is separated
by the ball's absolute position.

This is the intended compromise: it preserves the one-lookup architecture but gives
the hash locality and reusable features instead of hashing the whole context into a
nearly unique bucket.


In [ ]:
\
class BallTrapBrain:
    def __init__(
        self, alphabet, output, local_D=32, max_input_len=6,
        hash_radius=3, bounce=10, cap=10, direction_range=16
    ):
        self.alphabet = list(alphabet)
        self.index = {t: i for i, t in enumerate(self.alphabet)}
        self.output = list(output)

        self.local_D = int(local_D)
        self.max_input_len = int(max_input_len)
        self.hash_radius = int(hash_radius)
        self.D = self.local_D * self.max_input_len
        self.cap = int(cap)
        self.direction_range = int(direction_range)

        self.ball_energy = bounce * 2

        # Shared relative-position hash features.
        # Center offset is unused because s[token_i] is the center feature.
        self.hash_directions = np.random.randint(
            -self.direction_range,
            self.direction_range + 1,
            size=(2 * self.hash_radius + 1, len(self.alphabet)),
            dtype=np.int64,
        )
        self.hash_directions[self.hash_radius] = 0
        self.hash_start_positions = np.random.randint(
            0, self.local_D, size=len(self.alphabet), dtype=np.int64
        )

        # deflector = [destination, value]; value is an int (bounce energy) or str (hole symbol)
        self.deflectors_list = [
            [random.randint(0, self.D - 1), bounce] for _ in range(self.D)
        ]

    def start_nodes(self, input):
        toks = [t for t in TokenizeInput(input, self.alphabet) if t in self.index]
        ids = [self.index[t] for t in toks]
        nodes = []

        for i, tok_i in enumerate(ids):
            local = int(self.hash_start_positions[tok_i])
            for delta in range(-self.hash_radius, self.hash_radius + 1):
                if delta == 0:
                    continue
                j = i + delta
                if 0 <= j < len(ids):
                    local += int(self.hash_directions[delta + self.hash_radius, ids[j]])
            nodes.append(i * self.local_D + (local % self.local_D))

        return nodes

    def walk(self, node):
        out, dl, be, steps = [], self.deflectors_list, self.ball_energy, 0
        while be > 0 and steps < self.cap:
            dest, val = dl[node]
            dist = abs(dest - node)
            if isinstance(val, int):
                be += val - dist
            else:
                out.append(val)
                be += 1 - dist
            node, steps = dest, steps + 1
        return ''.join(out)

    def compute_input(self, input):
        return ''.join(self.walk(k) for k in self.start_nodes(input))

    def build_table(self):
        """T[k]: the string emitted from each bucket."""
        return [self.walk(k) for k in range(self.D)]

    def display_deflectors(self, n=20):
        for i, defl in enumerate(self.deflectors_list[:n]):
            print(f"Deflector {i}: {defl}")


def evaluate(brain, cases):
    exact = ratio = 0.0
    for inp, ans in cases:
        pred = brain.compute_input(inp)
        exact += pred == ans
        ratio += difflib.SequenceMatcher(None, pred, ans).ratio()
    return exact / len(cases), ratio / len(cases)


## The trainer

The table remains solved exactly by profiling each bucket to its empirical class
distribution / majority class.

The hash search is changed in three ways:

1. **Local relative features:** optimize `d[offset, token]` rather than one global
   `d[token]`. The same feature is reused at every absolute position.
2. **Slot-separated buckets:** each input position gets `local_D` buckets, so an
   `"8"` emitted by the ones slot cannot conflict with an `"8"` emitted by the tens
   slot merely because the strings happen to match.
3. **Reuse-aware tie-breaking:** after minimizing the primary profiled loss, candidate
   moves prefer fewer occupied `(bucket, class)` pairs. This explicitly favors
   reusing a learned feature instead of inventing another bucket for an equivalent
   class.

Coordinate search is still exact for each parameter: every allowed integer value is
evaluated and the best candidates are verified against the full loss.


In [ ]:
\
class HashTrainer:
    def __init__(
        self, brain, cases, loss='ce', resegment=True, max_len=None,
        seed=0, reuse_tiebreak=0.01
    ):
        self.brain = brain
        self.loss = loss
        self.do_reseg = resegment
        self.D = brain.D
        self.local_D = brain.local_D
        self.V = brain.direction_range
        self.R = brain.hash_radius
        self.max_len = min(max_len or brain.cap, brain.cap)
        self.rng = np.random.default_rng(seed)
        self.reuse_tiebreak = float(reuse_tiebreak)

        # ---- examples -> token ids + target strings
        self.ex_tokens, self.ex_target, dropped = [], [], 0
        out_set = set(brain.output)
        for inp, tgt in cases:
            toks = [brain.index[t] for t in TokenizeInput(inp, brain.alphabet)]
            if (
                not toks
                #or len(tgt) > len(toks) * self.max_len
                #or not set(tgt) <= out_set
                #or len(toks) > brain.max_input_len
            ):
                dropped += 1
                continue
            self.ex_tokens.append(toks)
            self.ex_target.append(tgt)

        # Identical token contexts cannot receive different outputs.
        by_ctx = {}
        for toks, tgt in zip(self.ex_tokens, self.ex_target):
            by_ctx.setdefault(tuple(toks), []).append(tgt)
        keep = {
            k: max(set(v), key=lambda t: (v.count(t), -v.index(t)))
            for k, v in by_ctx.items()
        }
        pairs = [
            (k, t)
            for k, t in zip(self.ex_tokens, self.ex_target)
            if keep[tuple(k)] == t
        ]
        ambiguous = len(self.ex_tokens) - len(pairs)
        self.ex_tokens = [k for k, _ in pairs]
        self.ex_target = [t for _, t in pairs]
        self.n_ex = len(self.ex_tokens)

        if dropped:
            print(f"[trainer] dropped {dropped} examples the brain cannot express")
        if ambiguous:
            print(
                f"[trainer] dropped {ambiguous} examples whose tokenized context "
                f"has conflicting targets"
            )

        # ---- classes = strings a single ball may emit
        self.cls_id, self.cls_str = {}, []
        self._reg("")
        for ch in brain.output:
            self._reg(ch)

        pieces = [
            self.canonical_split(t, len(k))
            for k, t in zip(self.ex_tokens, self.ex_target)
        ]
        for ps in pieces:
            for p in ps:
                self._reg(p)
        self.C = len(self.cls_str)

        # ---- flat position arrays
        lens = np.array([len(k) for k in self.ex_tokens], dtype=np.int64)
        self.starts = np.concatenate([[0], np.cumsum(lens)])
        self.NP = int(self.starts[-1])

        self.P_tok = np.concatenate(
            [np.asarray(k, dtype=np.int64) for k in self.ex_tokens]
        )
        self.P_i = np.concatenate(
            [np.arange(1, n + 1, dtype=np.int64) for n in lens]
        )
        self.P_pos = self.P_i - 1
        self.P_ex = np.repeat(np.arange(self.n_ex), lens)
        self.P_c = np.array(
            [self.cls_id[p] for ps in pieces for p in ps],
            dtype=np.int64,
        )

        # ---- hash parameters
        self.d = brain.hash_directions.copy()
        self.s = brain.hash_start_positions.copy()

        # ---- parameter occurrence lists
        #
        # s[token] affects the ball whose center token is that token.
        # d[offset, token] affects every destination ball that sees that token
        # at that relative offset inside its radius.
        occ_s, occ_d = {}, {}

        for e, toks in enumerate(self.ex_tokens):
            base = self.starts[e]
            L = len(toks)

            for i, tok in enumerate(toks):
                occ_s.setdefault(tok, []).append(base + i)

                for delta in range(-self.R, self.R + 1):
                     if delta == 0:
                         continue
                     j = i + delta
                     if 0 <= j < L:
                         key = (delta, toks[j])
                         occ_d.setdefault(key, []).append(base + i)

        self.occ_s = {
            tok: np.array(ps, dtype=np.int64)
            for tok, ps in occ_s.items()
        }
        self.occ_d = {
            key: np.array(ps, dtype=np.int64)
            for key, ps in occ_d.items()
        }

        self.s_params = sorted(self.occ_s)
        self.d_params = sorted(self.occ_d)

        self.vals_s = np.arange(self.local_D, dtype=np.int64)
        self.vals_d = np.arange(-self.V, self.V + 1, dtype=np.int64)

        self.flogf = np.zeros(self.NP + 2)
        k = np.arange(1, self.NP + 2, dtype=np.float64)
        self.flogf[1:] = k * np.log(k)

        self._rebuild()

    # ---------------------------------------------------------------- classes / split
    def _reg(self, piece):
        if piece not in self.cls_id:
            self.cls_id[piece] = len(self.cls_str)
            self.cls_str.append(piece)

    def canonical_split(self, t, L):
        """Right-align target characters onto the final balls."""
        if len(t) <= L:
            return [""] * (L - len(t)) + list(t)

        out, rest = [], t
        surplus = len(t) - L
        for _ in range(L):
            take = 1 + min(surplus, self.max_len - 1)
            surplus -= take - 1
            out.append(rest[:take])
            rest = rest[take:]
        return out

    # ---------------------------------------------------------------- hash + counts
    def addresses(self):
        h = np.empty(self.NP, dtype=np.int64)
        for e, toks in enumerate(self.ex_tokens):
            base = self.starts[e]
            L = len(toks)
            for i, tok_i in enumerate(toks):
                local = int(self.s[tok_i])
                for delta in range(-self.R, self.R + 1):
                    if delta == 0:
                        continue
                    j = i + delta
                    if 0 <= j < L:
                        local += int(
                            self.d[delta + self.R, toks[j]]
                        )
                h[base + i] = (
                    i * self.local_D + (local % self.local_D)
                )
        return h

    def _rebuild(self):
        self.h = self.addresses()
        self.N = np.zeros((self.D, self.C), dtype=np.int16)
        np.add.at(self.N, (self.h, self.P_c), 1)
        self.nk = self.N.sum(1)
        self.mx = self.N.max(1)

    def conflicts(self):
        return int((self.nk - self.mx).sum())

    def ce(self):
        return float(
            self.flogf[self.nk].sum() - self.flogf[self.N].sum()
        )

    def class_cells(self):
        return int(np.count_nonzero(self.N))

    def _primary(self):
        return self.conflicts() if self.loss == 'hard' else self.ce()

    def _remove(self, P):
        b = self.h[P]
        np.subtract.at(self.N, (b, self.P_c[P]), 1)
        u = np.unique(b)
        self.nk[u] = self.N[u].sum(1)
        self.mx[u] = self.N[u].max(1)

    def _insert(self, P, hn):
        self.h[P] = hn
        np.add.at(self.N, (hn, self.P_c[P]), 1)
        u = np.unique(hn)
        self.nk[u] = self.N[u].sum(1)
        self.mx[u] = self.N[u].max(1)

    def _marginal_primary(self, idx, c):
        """Exact marginal primary loss for adding one position."""
        Nc = self.N[idx, c]
        if self.loss == 'hard':
            return (Nc < self.mx[idx]).astype(np.float64)

        n = self.nk[idx]
        return (
            (self.flogf[n + 1] - self.flogf[n])
            - (self.flogf[Nc + 1] - self.flogf[Nc])
        )

    def _marginal_class_cells(self, idx, c):
        """Adding a position changes class-cell count only when that cell is empty."""
        return (self.N[idx, c] == 0).astype(np.float64)

    # ---------------------------------------------------------------- exact coordinate move
    def _move(self, P, cur, vals, ntry=8):
        P = np.asarray(P, dtype=np.int64)
        old = self.h[P].copy()
        before_primary = self._primary()
        before_cells = self.class_cells()

        self._remove(P)

        # All local feature parameters shift affected addresses by the same delta.
        base = (old - cur) % self.local_D
        pos_off = self.P_pos[P] * self.local_D
        cls = self.P_c[P][:, None]

        primary_cost = np.empty(len(vals))
        reuse_cost = np.empty(len(vals))

        # Evaluate candidate values in chunks so the (positions x values) block
        # never exceeds ~4,000,000 cells. Each candidate's cost is independent,
        # so this gives identical results to building the full matrix at once.
        chunk = max(1, 4000000 // max(1, len(P)))
        for a in range(0, len(vals), chunk):
            v = vals[a:a + chunk]
            idx = (base[:, None] + v[None, :]) % self.local_D + pos_off[:, None]
            primary_cost[a:a + chunk] = self._marginal_primary(idx, cls).sum(0)
            # Small secondary preference for reusing an existing (bucket,class) pair.
            reuse_cost[a:a + chunk] = self._marginal_class_cells(idx, cls).sum(0)

        score = primary_cost + self.reuse_tiebreak * reuse_cost
        score = score + self.rng.random(len(vals)) * 1e-7

        for vi in np.argsort(score)[:ntry]:
            hn = (
                (base + vals[vi]) % self.local_D
                + self.P_pos[P] * self.local_D
            )
            self._insert(P, hn)

            after_primary = self._primary()
            after_cells = self.class_cells()

            primary_better = after_primary < before_primary - 1e-12
            primary_tied = abs(after_primary - before_primary) <= 1e-12
            accept = primary_better or (primary_tied and after_cells <= before_cells)

            if accept:
                return int(vals[vi])

            self._remove(P)

        self._insert(P, old)
        return int(cur)

    def sweep(self, ntry=8):
        params = (
            [('s', p) for p in self.s_params]
            + [('d', p) for p in self.d_params]
        )
        self.rng.shuffle(params)

        for kind, p in params:
            if kind == 's':
                P = self.occ_s[p]
                self.s[p] = self._move(
                    P, self.s[p], self.vals_s, ntry=ntry
                )
            else:
                delta, tok = p
                P = self.occ_d[p]
                row = delta + self.R
                self.d[row, tok] = self._move(
                    P, self.d[row, tok], self.vals_d, ntry=ntry
                )

    # ---------------------------------------------------------------- latent split
    def table_strings(self):
        mode = self.N.argmax(1)
        return [self.cls_str[c] for c in mode]

    def resegment(self):
        """Hard-EM split update; the loss cannot increase before re-profiling the table."""
        T = self.table_strings()
        changed = 0

        for e in range(self.n_ex):
            a, b = self.starts[e], self.starts[e + 1]
            L = b - a
            t = self.ex_target[e]
            n = len(t)

            tab = [T[self.h[p]] for p in range(a, b)]
            cur = [self.cls_str[c] for c in self.P_c[a:b]]
            if tab == cur:
                continue

            INF = 1e9
            dp = np.full((L + 1, n + 1), INF)
            bk = [[None] * (n + 1) for _ in range(L + 1)]
            dp[0, 0] = 0.0

            for i in range(L):
                for o in range(n + 1):
                    if dp[i, o] >= INF:
                        continue
                    for o2 in range(o, min(n, o + self.max_len) + 1):
                        piece = t[o:o2]
                        if piece not in self.cls_id:
                            continue
                        c = (
                            dp[i, o]
                            + (piece != tab[i])
                            + 1e-3 * (piece != cur[i])
                        )
                        if c < dp[i + 1, o2]:
                            dp[i + 1, o2] = c
                            bk[i + 1][o2] = (o, piece)

            if bk[L][n] is None:
                continue

            o, new = n, []
            for i in range(L, 0, -1):
                o, piece = bk[i][o]
                new.append(piece)
            new.reverse()

            if new != cur:
                self.P_c[a:b] = [
                    self.cls_id[piece] for piece in new
                ]
                changed += 1

        self._rebuild()
        return changed

    # ---------------------------------------------------------------- metrics
    def table_exact_match(self):
        T = self.table_strings()
        ok = []
        for e in range(self.n_ex):
            a, b = self.starts[e], self.starts[e + 1]
            ok.append(
                ''.join(T[self.h[p]] for p in range(a, b))
                == self.ex_target[e]
            )
        return float(np.mean(ok))

    def feature_reuse(self):
        occupied = int(np.count_nonzero(self.N))
        return self.NP - occupied

    # ---------------------------------------------------------------- fit
    def fit(self, rounds=10, sweeps=10, ntry=8, verbose=True):
        t0 = time.time()
        best = (
            self._primary(),
            self.class_cells(),
            self.d.copy(),
            self.s.copy(),
            self.P_c.copy(),
        )

        if verbose:
            print(
                f"init: primary={self._primary():.4f}  "
                f"ce={self.ce():.2f}  "
                f"class-cells={self.class_cells()}"
            )

        for r in range(rounds):
            stalled = 0
            prev = (self._primary(), self.class_cells())
            for sw in range(sweeps):
                self.sweep(ntry=ntry)
                now = (self._primary(), self.class_cells())

                if verbose:
                    print(
                        f"  round {r} sweep {sw}: "
                        f"primary={now[0]:.4f}  "
                        f"ce={self.ce():.2f}  "
                        f"class-cells={now[1]}"
                    )

                if now < best[:2]:
                    best = (
                        now[0], now[1],
                        self.d.copy(),
                        self.s.copy(),
                        self.P_c.copy(),
                    )

                if now == prev:
                     stalled += 1
                     if stalled >= 2:
                         break
                else:
                    stalled = 0
                prev = now

            if self.do_reseg:
                ch = self.resegment()
                if verbose and ch:
                    print(
                        f"  round {r}: resegmented {ch} examples; "
                        f"primary={self._primary():.4f}"
                    )

            if self.loss == 'hard' and self.conflicts() == 0:
                break

        self.d, self.s, self.P_c = best[2], best[3], best[4]
        self._rebuild()

        if verbose:
            print(
                f"done: primary={self._primary():.4f}  "
                f"conflicts={self.conflicts()}  "
                f"table-train-exact={self.table_exact_match():.4f}  "
                f"reuse={self.feature_reuse()}  "
                f"[{time.time() - t0:.1f}s]"
            )
        return self

    # ---------------------------------------------------------------- compile to real deflectors
    def compile(self, verbose=True):
        """
        Write the learned local hash and the profiled table into the physical brain.

        A bucket whose string is 'abc' becomes a -> b -> c -> sink on free nodes.
        """
        br = self.brain
        BIG = 10 ** 9

        br.hash_directions = self.d.copy()
        br.hash_start_positions = self.s.copy()

        T = self.table_strings()
        used = self.nk > 0
        free = ~used

        # Deletion-only predecessor/successor indexes. Path compression skips
        # occupied runs while preserving the original nearest-node allocation.
        right = list(range(len(free) + 1))  # len(free) is the right sentinel
        left = list(range(len(free) + 1))   # 0 sentinel; node x is stored at x+1
        for x in range(len(free) - 1, -1, -1):
            if not free[x]:
                right[x] = right[x + 1]
        for x in range(len(free)):
            if not free[x]:
                left[x + 1] = left[x]

        def find(parent, x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def occupy(x):
            free[x] = False
            right[x] = find(right, x + 1)
            left[x + 1] = find(left, x)

        def nearest_free(x):
            nonlocal free, dl
            x = int(x)
            lo = find(left, x) - 1
            hi = find(right, x + 1)
            if lo >= 0 and (hi == len(free) or x - lo <= hi - x):
                return lo
            if hi < len(free):
                return hi
            # Same expansion size and first new node as the original compiler.
            old_size = len(free)
            extra_needed = max(100, int(old_size * 0.1))
            free = np.concatenate([free, np.ones(extra_needed, dtype=bool)])
            dl.extend([[0, -BIG] for _ in range(extra_needed)])
            # The former right sentinel becomes the first appended free node.
            right.extend(range(old_size + 1, len(free) + 1))
            left.extend(range(old_size + 1, len(free) + 1))
            br.D = len(dl)
            self.D = len(dl)
            return old_size

        dl = [[k, -BIG] for k in range(self.D)]

        # Create sink on an empty node
        if not np.any(free):
            nearest_free(0)
        sink = int(np.argmax(free))
        occupy(sink)

        need = 1
        chain_nodes = 0
        max_hop = 0

        for k in np.flatnonzero(used):
            S = T[k]
            if not S:
                continue

            nodes = [int(k)]
            for _ in S[1:]:
                u = nearest_free(nodes[-1])
                occupy(u)
                nodes.append(u)
                chain_nodes += 1

            for node, ch, nxt in zip(
                nodes, S, nodes[1:] + [sink]
            ):
                dl[node] = [int(nxt), ch]

            cumulative_cost = 0
            for a, b in zip(nodes, nodes[1:]):
                cumulative_cost += abs(b - a) - 1
                max_hop = max(max_hop, abs(b - a))
            need = max(need, cumulative_cost + 1)

        br.deflectors_list = dl
        br.ball_energy = max(br.ball_energy, need)

        if verbose:
            print(
                f"compiled: start-buckets={int(used.sum())}  "
                f"chain-nodes={chain_nodes}  "
                f"longest-emission={max(len(s) for s in T)}  "
                f"longest-hop={max_hop}  "
                f"ball-energy={br.ball_energy}"
            )
        return br


def train_with_restarts(cases, usedBrain=False):
    """
    Multi-start discrete optimization. Each restart gets a fresh random brain and trainer;
    the best primary loss, then lowest class-cell count, is kept.
    """
    best = None

    for r in range(CFG['restarts']):
        seed = CFG['seed'] + r
        random.seed(seed)
        np.random.seed(seed)

        brain = BallTrapBrain(
            alphabet,
            output,
            local_D=CFG['local_D'],
            max_input_len=max_input_len,
            hash_radius=CFG['hash_radius'],
            bounce=CFG['bounce'],
            cap=CFG['cap'],
            direction_range=CFG['direction_range'],
        )
        if usedBrain:
            brain.hash_directions = usedBrain.hash_directions
            brain.hash_start_positions = usedBrain.hash_start_positions
            brain.deflectors_list = usedBrain.deflectors_list
            brain.ball_energy = usedBrain.ball_energy
            brain.cap = usedBrain.cap

        trainer = SparseGPUAlignedHashTrainer(
            brain,
            cases,
            loss=CFG['loss'],
            resegment=True,
            seed=seed,
            reuse_tiebreak=CFG['reuse_tiebreak'],
            free_reseg=CFG['free_reseg'],
            copy_split=CFG['copy_split'],
            sparse_mode=CFG['sparse_mode'],
        )
        trainer.fit(
            rounds=CFG['rounds'],
            sweeps=CFG['sweeps'],
            ntry=CFG['ntry'],
            verbose=False,
        )

        score = (trainer._primary(), trainer.class_cells())
        print(
            f"restart {r}: primary={score[0]:.4f}  "
            f"conflicts={trainer.conflicts()}  "
            f"train-table-exact={trainer.table_exact_match():.4f}  "
            f"reuse={trainer.feature_reuse()}"
        )

        if best is None or score < best[0]:
            best = (score, trainer)

    print(
        f"selected: primary={best[0][0]:.4f}  "
        f"conflicts={best[1].conflicts()}  "
        f"train-table-exact={best[1].table_exact_match():.4f}"
    )
    return best[1]

In [ ]:
# ---- FAST TRAINER ----------------------------------------------------------
# Drop-in subclass of HashTrainer. Identical search/results, but the loss and class-cell
# totals are updated incrementally (only the buckets a move touches) instead of
# re-scanning the whole D x C count table on every candidate. Run after the HashTrainer cell.
class FastHashTrainer(HashTrainer):
    """Same search as HashTrainer, but the loss / class-cell totals are updated
    incrementally (O(positions touched)) instead of re-scanning the whole D x C table."""

    def __init__(self, *a, dedupe_vals=True, **kw):
        super().__init__(*a, **kw)
        if dedupe_vals:   # offsets are only used mod local_D -> drop duplicate residues
            rep = {}
            for v in self.vals_d:
                rep.setdefault(int(v) % self.local_D, int(v))
            self.vals_d = np.array(sorted(rep.values()), dtype=np.int64)

    # ---- state ------------------------------------------------------------
    def _rebuild(self):
        self.h = self.addresses()
        self.N = np.zeros((self.D, self.C), dtype=np.int16)
        np.add.at(self.N, (self.h, self.P_c), 1)
        self.Nf = self.N.reshape(-1)                  # flat view, same memory
        self.nk = self.N.sum(1).astype(np.int64)
        self.mx = self.N.max(1)
        self._resync()

    def _resync(self):
        nz = np.flatnonzero(self.Nf)                  # N is very sparse: sum only nonzero cells
        self._F = float(self.flogf[self.Nf[nz]].sum())
        self._NK = float(self.flogf[self.nk].sum())
        self._cells = int(np.count_nonzero(self.N))

    def ce(self):
        if self.loss != 'ce':
            return super().ce()
        return self._NK - self._F

    def class_cells(self):
        return self._cells if self.loss == 'ce' else super().class_cells()

    def ce_full(self):                                # for verification only
        return float(self.flogf[self.nk].sum() - self.flogf[self.N].sum())

    # ---- incremental add / remove -------------------------------------------
    def _apply(self, P, b, sign):
        key = b * self.C + self.P_c[P]
        uk, cnt = np.unique(key, return_counts=True)
        old = self.Nf[uk].astype(np.int64)
        new = old + sign * cnt
        self.Nf[uk] = new
        dF = self.flogf[new].sum() - self.flogf[old].sum()
        dcells = int(np.count_nonzero(new) - np.count_nonzero(old))
        ub, bc = np.unique(b, return_counts=True)
        oldk = self.nk[ub]
        newk = oldk + sign * bc
        self.nk[ub] = newk
        dNK = self.flogf[newk].sum() - self.flogf[oldk].sum()
        self._F += dF; self._NK += dNK; self._cells += dcells
        return float(dNK - dF), dcells

    def sweep(self, ntry=8):
        self._resync()
        super().sweep(ntry=ntry)
        self._resync()

    # ---- exact coordinate move (same logic, O(touched) bookkeeping) ----------
    def _move(self, P, cur, vals, ntry=8):
        if self.loss != 'ce':
            return super()._move(P, cur, vals, ntry)
        P = np.asarray(P, dtype=np.int64)
        old = self.h[P].copy()
        d_rm, c_rm = self._apply(P, old, -1)

        base = (old - cur) % self.local_D
        pos_off = self.P_pos[P] * self.local_D
        cls = self.P_c[P][:, None]
        primary_cost = np.empty(len(vals))
        reuse_cost = np.empty(len(vals))
        chunk = max(1, 4000000 // max(1, len(P)))
        for a in range(0, len(vals), chunk):
            v = vals[a:a + chunk]
            idx = (base[:, None] + v[None, :]) % self.local_D + pos_off[:, None]
            primary_cost[a:a + chunk] = self._marginal_primary(idx, cls).sum(0)
            reuse_cost[a:a + chunk] = self._marginal_class_cells(idx, cls).sum(0)

        score = primary_cost + self.reuse_tiebreak * reuse_cost
        score = score + self.rng.random(len(vals)) * 1e-7

        for vi in np.argsort(score)[:ntry]:
            hn = (base + vals[vi]) % self.local_D + pos_off
            self.h[P] = hn
            d_in, c_in = self._apply(P, hn, +1)
            dp, dc = d_rm + d_in, c_rm + c_in
            if dp < -1e-9 or (abs(dp) <= 1e-9 and dc <= 0):
                return int(vals[vi])
            self._apply(P, hn, -1)
        self.h[P] = old
        self._apply(P, old, +1)
        return int(cur)


## Alignment patch: `AlignedHashTrainer`

`canonical_split` ties chunk boundaries to `len(prompt)` / `len(answer)`, so the same slot with the same local context is asked to emit different text in different examples.
Two fixes, both driven by a small segmentation DP (split the answer into exactly `L` consecutive pieces of at most `cap` characters; empty pieces allowed):

1. **`free_reseg`** - `resegment()` may choose *any* substring as a piece (new pieces are registered as classes on the fly) instead of only already-known classes.
2. **`copy_split`** - content-anchored initial split: a ball whose center token is a number is rewarded for emitting exactly that number, and other cuts prefer word edges.

Caveats: `C` grows, so `D x C` memory grows; copy alignment is monotone (ball order = output order); and with slot-separated buckets a copy has to be learned per slot.


### Exact initial-split and resegmentation acceleration

`copy_split=True` and `free_reseg=True` now use a compiled numeric dynamic program.
If needed, run `%pip install numba` in a separate cell, restart the kernel, and run this notebook from the top. The first segmentation call compiles the kernel; subsequent calls are fast. Without Numba, the original callback implementation remains available and is used automatically with a warning.

The optimization precomputes substring identities and Python Unicode string predicates, uses rolling cost rows, and skips paths that cannot reach the required final endpoint. It preserves float64 operation order, strict `<` tie-breaking, and sequential registration of new classes. No greedy approximation, integer cost rounding, example parallelism, or random-number consumption is introduced. `free_reseg=False` continues to use its original implementation. GPU/sparse subclasses inherit the accelerated methods.

Validation: 3,000 randomized exact-piece comparisons (including Unicode, empty and impossible cases) passed. Initial trainer state and four alternating free-resegmentation/coordinate-sweep rounds matched exactly, including class order, counts, hashes, parameters, and loss. Tests ran on the CPU AlignedHashTrainer; GPU execution and the full training dataset were not available for validation.

Synthetic warmed-up benchmark (180 input positions, 168 target characters, max piece length 10; preprocessing included): initial segmentation 0.134 s → 0.00158 s (~85×), free resegmentation 0.128 s → 0.00127 s (~101×). These measure one segmentation, not complete initialization, rebuilding, or total training. Runtime depends on input shape and hardware. Worst-case DP time remains O(L n max_len); no exact linear-time claim is made.


In [ ]:
"""
AlignedHashTrainer: content-anchored answer alignment + live progress bars.
Run after the FastHashTrainer cell.

  AlignedHashTrainer(brain, cases, ..., free_reseg=True)    # option 1: resegment into ANY substring
  AlignedHashTrainer(brain, cases, ..., copy_split=True)    # option 2: content-anchored initial split
  AlignedHashTrainer(brain, cases, ..., progress=True, label="restart 1/3")   # progress bars
"""
import numpy as np

try:
    from tqdm.auto import tqdm as _tqdm      # notebook widget bar in Jupyter/Kaggle/Colab, text bar elsewhere
except Exception:                            # tqdm missing -> run silently
    _tqdm = None

NOVEL_PIECE_COST = 0.01   # prefer pieces that already exist as classes (keeps C, and D x C, small)
MIDWORD_CUT_COST = 0.02   # prefer cutting the answer at word edges, not inside a word
COPY_REWARD      = 1.0    # reward for ball i emitting exactly its own digit token


class _NullBar:
    n = 0
    total = 0
    def update(self, *a, **k): pass
    def set_postfix(self, *a, **k): pass
    def set_description(self, *a, **k): pass
    def refresh(self): pass
    def close(self): pass


# Exact segmentation acceleration. No fastmath: rounding and first-winner ties matter.
try:
    from numba import njit as _segment_njit
except ImportError:
    _segment_njit = None
    import warnings
    warnings.warn("Numba is unavailable: segmentation uses the original Python DP. "
                  "Run %pip install numba, then restart and run all cells for acceleration.")


def _segment_numeric(L, n, cap, piece_ids, strip_ids, cut, known,
                     first, second, mode, novel_cost, cut_cost, copy_reward):
    INF = 1e18
    prev = np.full(n + 1, INF)
    prev[0] = 0.0
    bk = np.zeros((L + 1, n + 1), dtype=np.int32)
    for i in range(L):
        nxt = np.full(n + 1, INF)
        # Only paths that can still reach the final endpoint are needed.
        for o in range(max(0, n - (L - i) * cap), min(n, i * cap) + 1):
            base = prev[o]
            if base >= INF:
                continue
            for o2 in range(max(o, n - (L - i - 1) * cap), min(n, o + cap) + 1):
                pid = piece_ids[o, o2 - o]
                if mode == 0:
                    c = 1e-3 * (pid != first[i])
                    if o2 > o and cut[o2]:
                        c += cut_cost
                    if second[i] >= 0 and strip_ids[pid] == second[i]:
                        c -= copy_reward
                else:
                    c = ((pid != first[i]) + 1e-3 * (pid != second[i])
                         + (novel_cost if not known[pid] else 0.0))
                c = base + c
                if c < nxt[o2]:
                    nxt[o2] = c
                    bk[i + 1, o2] = o
        prev = nxt
    if prev[n] >= INF:
        return None
    edges = np.empty(L + 1, dtype=np.int64)
    edges[L] = n
    for i in range(L, 0, -1):
        edges[i - 1] = bk[i, edges[i]]
    return edges


if _segment_njit is not None:
    _segment_numeric = _segment_njit(fastmath=False)(_segment_numeric)


class AlignedHashTrainer(FastHashTrainer):
    def __init__(self, *a, free_reseg=True, copy_split=False, progress=True, label="", **kw):
        # progress state must exist before super().__init__ (resegment/_move can run during init)
        self.progress = bool(progress) and _tqdm is not None
        self.label = label or "training"
        self._bar = _NullBar()
        self._round = 0
        self._sw_done = 0
        self._planned_sweeps = 0
        self._n_params = 0
        super().__init__(*a, **kw)
        self.free_reseg = free_reseg
        self.copy_stats = None
        if copy_split:
            self._resplit_initial()

    # ------------------------------------------------------------------ progress plumbing
    def _tq(self, it, **kw):
        return _tqdm(it, mininterval=0.5, **kw) if self.progress else it

    def fit(self, rounds=10, sweeps=10, ntry=8, verbose=True):
        """One bar over every coordinate move of every sweep of every round.
        The bar's total shrinks when a round stops early (stalled) so the ETA stays honest."""
        if not self.progress:
            return super().fit(rounds=rounds, sweeps=sweeps, ntry=ntry, verbose=verbose)
        self._n_params = len(self.s_params) + len(self.d_params)
        self._planned_sweeps = sweeps
        self._round, self._sw_done = 0, 0
        self._bar = _tqdm(total=rounds * sweeps * self._n_params, desc=self.label,
                          unit="coord", unit_scale=True, mininterval=0.5)
        self._bar.set_postfix(round=f"1/{rounds}", sweep=f"0/{sweeps}")
        self._rounds = rounds
        try:
            return super().fit(rounds=rounds, sweeps=sweeps, ntry=ntry, verbose=verbose)
        finally:
            self._bar.set_postfix(ce=f"{self._primary():.1f}", conflicts=self.conflicts(), done=True)
            self._bar.total = self._bar.n          # early stop (stalled / no conflicts): show 100%
            self._bar.refresh()
            self._bar.close()
            self._bar = _NullBar()

    def _move(self, P, cur, vals, ntry=8):
        out = super()._move(P, cur, vals, ntry=ntry)
        self._bar.update(1)
        return out

    def sweep(self, ntry=8):
        super().sweep(ntry=ntry)
        self._sw_done += 1
        self._bar.set_postfix(round=f"{self._round + 1}/{getattr(self, '_rounds', '?')}",
                              sweep=f"{self._sw_done}/{self._planned_sweeps}",
                              ce=f"{self._primary():.1f}", conflicts=self.conflicts())

    def _end_round(self):
        """Called when a round's sweeps are over (possibly early): drop the skipped sweeps from the total."""
        skipped = max(self._planned_sweeps - self._sw_done, 0)
        if skipped and self._bar.total:
            self._bar.total = max(self._bar.total - skipped * self._n_params, self._bar.n)
            self._bar.refresh()

    def _start_round(self):
        self._round += 1
        self._sw_done = 0

    # ------------------------------------------------------------------ shared DP
    def _segment(self, t, L, piece_cost):
        """Min-cost split of t into exactly L consecutive pieces (each <= max_len, may be empty)."""
        n, INF = len(t), 1e18
        dp = np.full((L + 1, n + 1), INF)
        bk = np.zeros((L + 1, n + 1), dtype=np.int32)
        dp[0, 0] = 0.0
        for i in range(L):
            for o in range(n + 1):
                base = dp[i, o]
                if base >= INF:
                    continue
                for o2 in range(o, min(n, o + self.max_len) + 1):
                    c = base + piece_cost(i, o, o2)
                    if c < dp[i + 1, o2]:
                        dp[i + 1, o2] = c
                        bk[i + 1, o2] = o
        if dp[L, n] >= INF:
            return None
        out, o = [], n
        for i in range(L, 0, -1):
            o0 = int(bk[i, o])
            out.append(t[o0:o])
            o = o0
        return out[::-1]

    def _segment_exact(self, t, L, first, second, mode, fallback):
        """Precompute Python string semantics once, then run the exact numeric DP.

        mode 0: canonical pieces and stripped input tokens (initial split).
        mode 1: table pieces and current pieces (free resegmentation).
        The generic callback DP remains the fallback when Numba is unavailable.
        """
        if _segment_njit is None:
            return self._segment(t, L, fallback)
        n, cap, L = len(t), min(self.max_len, len(t)), int(L)
        if n > L * cap:
            return None
        ids, pieces = {}, []
        def intern(p):
            if p not in ids:
                ids[p] = len(pieces)
                pieces.append(p)
            return ids[p]
        grid = np.zeros((n + 1, cap + 1), dtype=np.int64)
        for o in range(n + 1):
            for size in range(min(cap, n - o) + 1):
                grid[o, size] = intern(t[o:o + size])
        first_ids = np.array([intern(p) for p in first], dtype=np.int64)
        if mode == 0:
            second_ids = np.array([intern(w) if w.isdigit() else -1
                                   for w in second], dtype=np.int64)
            # Intern stripped forms before allocating arrays; Unicode semantics
            # deliberately remain Python's str.strip/isdigit/isalnum semantics.
            stripped = [intern(p.strip()) for p in list(pieces)]
            strip_ids = np.zeros(len(pieces), dtype=np.int64)
            strip_ids[:len(stripped)] = stripped
        else:
            second_ids = np.array([intern(p) for p in second], dtype=np.int64)
            strip_ids = np.zeros(len(pieces), dtype=np.int64)
        known = np.array([p in self.cls_id for p in pieces], dtype=np.bool_)
        cut = np.zeros(n + 1, dtype=np.bool_)
        for o in range(1, n):
            cut[o] = t[o - 1].isalnum() and t[o].isalnum()
        edges = _segment_numeric(L, n, cap, grid, strip_ids, cut, known,
                                 first_ids, second_ids, mode, NOVEL_PIECE_COST,
                                 MIDWORD_CUT_COST, COPY_REWARD)
        if edges is None:
            return None
        return [t[edges[i]:edges[i + 1]] for i in range(L)]

    def _ids(self, pieces):
        for p in pieces:
            self._reg(p)                      # registers unseen pieces on the fly
        return [self.cls_id[p] for p in pieces]

    # ------------------------------------------------------------------ option 1
    def resegment(self):
        """Hard-EM split update where pieces are ANY substring (<= max_len), not just known classes."""
        self._end_round()
        self._bar.set_description(f"{self.label} (resegmenting)")
        try:
            if not self.free_reseg:
                return super().resegment()
            T = self.table_strings()
            changed = 0
            for e in self._tq(range(self.n_ex), desc="resegment", leave=False, unit="ex"):
                a, b = self.starts[e], self.starts[e + 1]
                L, t = b - a, self.ex_target[e]
                tab = [T[self.h[p]] for p in range(a, b)]
                cur = [self.cls_str[c] for c in self.P_c[a:b]]
                if tab == cur:
                    continue

                def cost(i, o, o2, t=t, tab=tab, cur=cur):
                    p = t[o:o2]
                    return ((p != tab[i])
                            + 1e-3 * (p != cur[i])
                            + (NOVEL_PIECE_COST if p not in self.cls_id else 0.0))

                new = self._segment_exact(t, L, tab, cur, 1, cost)
                if new is not None and new != cur:
                    self.P_c[a:b] = self._ids(new)
                    changed += 1
            self.C = len(self.cls_str)        # N is (D x C): _rebuild picks up the new C
            self._rebuild()
            return changed
        finally:
            self._bar.set_description(self.label)
            self._start_round()

    # ------------------------------------------------------------------ option 2
    def _resplit_initial(self):
        """Replace the length-based initial split by a content-anchored one:
        balls whose center token is a number may emit exactly that number (a copy),
        and every other cut prefers word edges."""
        alpha, copied, eligible = self.brain.alphabet, 0, 0
        for e in self._tq(range(self.n_ex), desc=f"{self.label}: initial split", leave=False, unit="ex"):
            a, b = self.starts[e], self.starts[e + 1]
            L, t = b - a, self.ex_target[e]
            n = len(t)
            words = [alpha[k].strip() for k in self.ex_tokens[e]]
            canon = self.canonical_split(t, L)

            def cost(i, o, o2, t=t, n=n, words=words, canon=canon):
                p = t[o:o2]
                c = 1e-3 * (p != canon[i])
                if o2 > o and 0 < o2 < n and t[o2 - 1].isalnum() and t[o2].isalnum():
                    c += MIDWORD_CUT_COST
                w = words[i]
                if w.isdigit() and p.strip() == w:
                    c -= COPY_REWARD
                return c

            new = self._segment_exact(t, L, canon, words, 0, cost)
            if new is None:
                continue
            self.P_c[a:b] = self._ids(new)
            for i, w in enumerate(words):
                if w.isdigit():
                    eligible += 1
                    copied += (new[i].strip() == w)
        self.C = len(self.cls_str)
        self._rebuild()
        self.copy_stats = (copied, eligible)

## Parallel scoring: `GPUAlignedHashTrainer`

Same search and acceptance rule as `AlignedHashTrainer`, but the candidate scoring inside each
coordinate move (the `positions x candidate values` block) runs on every available GPU, with the
candidate values split across them. With no GPU it runs on all CPU cores through torch.

* Each GPU holds a copy of the `D x C` count table and is kept in sync when a move is applied.
* `addresses()` is vectorized with numpy instead of a per-token Python loop.
* `resegment()` and the `_rebuild` bookkeeping are still single-threaded numpy/Python.

Options: `devices=['cuda:0','cuda:1','cpu']`, `shares=[1,1,0.15]`, `gpu_cells=2**24` (lower it on out-of-memory).

In [ ]:
# ---- PARALLEL TRAINER ------------------------------------------------------
# Run AFTER the AlignedHashTrainer cell. Drop-in subclass: same search, same
# acceptance rule, same results -- only the candidate scoring inside _move is moved
# onto every GPU (split across them) or, with no GPU, onto all CPU cores via torch.
#
#   trainer = GPUAlignedHashTrainer(brain, cases, ...)            # auto: all GPUs, else CPU
#   trainer = GPUAlignedHashTrainer(..., devices=['cuda:0'])      # pick devices
#   trainer = GPUAlignedHashTrainer(..., devices=['cuda:0','cuda:1','cpu'],
#                                        shares=[1, 1, 0.15])     # CPU takes a small slice too
#
# In train_with_restarts, change `AlignedHashTrainer(` to `GPUAlignedHashTrainer(`.
import os
import numpy as np
import torch


class GPUAlignedHashTrainer(AlignedHashTrainer):
    def __init__(self, *a, devices=None, shares=None,
                 gpu_cells=2 ** 24, cpu_cells=2 ** 22, **kw):
        self._mirror_ready = False
        self._dev_req, self._share_req = devices, shares
        self._gpu_cells, self._cpu_cells = int(gpu_cells), int(cpu_cells)
        self._dev = []
        torch.set_num_threads(os.cpu_count() or 1)       # torch-CPU ops use every core
        super().__init__(*a, **kw)
        self._upload_state()
        self._mirror_ready = True

    # ------------------------------------------------------------ vectorized hash
    def addresses(self):
        """Same values as the per-token Python loop, computed with numpy in one pass."""
        R, Ld = self.R, self.local_D
        tok, pos = self.P_tok, self.P_pos
        lens = np.diff(self.starts)
        Lp = np.repeat(lens, lens)
        ar = np.arange(self.NP)
        local = self.s[tok].astype(np.int64)
        for delta in range(-R, R + 1):
            if delta == 0:
                continue
            j = pos + delta
            valid = (j >= 0) & (j < Lp)
            nb = tok[np.clip(ar + delta, 0, self.NP - 1)]
            local += np.where(valid, self.d[delta + R, nb], 0)
        return pos * Ld + (local % Ld)

    # ------------------------------------------------------------ device mirrors
    def _upload_state(self):
        """(Re)build the device copies of N / nk / g. CPU 'devices' share memory with numpy."""
        self._dev = []
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        req = self._dev_req
        if req is None:
            req = ([f'cuda:{i}' for i in range(torch.cuda.device_count())]
                   if torch.cuda.is_available() else ['cpu'])
        shares = self._share_req if self._share_req is not None else [1.0] * len(req)

        g = (self.flogf[1:self.NP + 2] - self.flogf[:self.NP + 1]).astype(np.float64)
        need = self.D * self.C * 2 + self.D * 8 + g.nbytes
        kept = []
        for name, sh in zip(req, shares):
            dev = torch.device(name)
            if dev.type == 'cuda':
                free = torch.cuda.mem_get_info(dev)[0]
                if free < need + self._gpu_cells * 64:
                    print(f"[parallel] {name}: needs ~{(need + self._gpu_cells * 64) / 1e9:.1f} GB, "
                          f"only {free / 1e9:.1f} GB free -> skipped")
                    continue
                Nf = torch.from_numpy(self.Nf).to(dev)
                nk = torch.from_numpy(self.nk).to(dev)
                gt = torch.from_numpy(g).to(dev)
                cells = self._gpu_cells
            else:
                Nf, nk, gt = torch.from_numpy(self.Nf), torch.from_numpy(self.nk), torch.from_numpy(g)
                cells = self._cpu_cells
            kept.append(dict(name=name, dev=dev, Nf=Nf, nk=nk, g=gt, cells=cells, share=float(sh)))
        if not kept:
            print("[parallel] no usable device -> torch on CPU")
            kept = [dict(name='cpu', dev=torch.device('cpu'), Nf=torch.from_numpy(self.Nf),
                         nk=torch.from_numpy(self.nk), g=torch.from_numpy(g),
                         cells=self._cpu_cells, share=1.0)]
        self._dev = kept
        if not getattr(self, '_announced', False):
            self._announced = True
            print("[parallel] scoring on: " + ", ".join(
                f"{d['name']} (share {d['share']:g})" for d in kept)
                + f"  | count table {self.D}x{self.C} int16 = {self.D * self.C * 2 / 1e9:.2f} GB per GPU")

    def _rebuild(self):
        super()._rebuild()
        if getattr(self, '_mirror_ready', False):
            self._upload_state()

    def _apply(self, P, b, sign):
        out = super()._apply(P, b, sign)
        if getattr(self, '_mirror_ready', False):
            P = np.asarray(P)
            key = np.unique(np.asarray(b) * self.C + self.P_c[P])
            ub = np.unique(b)
            kv, nv = self.Nf[key], self.nk[ub]
            for st in self._dev:
                if st['dev'].type == 'cpu':
                    continue                              # zero-copy view, already current
                k_t = torch.as_tensor(key, device=st['dev'])
                u_t = torch.as_tensor(ub, device=st['dev'])
                st['Nf'][k_t] = torch.as_tensor(kv, device=st['dev'])
                st['nk'][u_t] = torch.as_tensor(nv, device=st['dev'])
        return out

    # ------------------------------------------------------------ parallel scoring
    def _score_on(self, st, base, pos_off, cls, vals):
        dev = st['dev']
        b = torch.as_tensor(base, device=dev)
        po = torch.as_tensor(pos_off, device=dev)
        c = torch.as_tensor(cls, device=dev)[:, None]
        v = torch.as_tensor(vals, device=dev)
        n, m, C, Ld = b.numel(), v.numel(), self.C, self.local_D
        out_p = torch.empty(m, dtype=torch.float64, device=dev)
        out_r = torch.empty(m, dtype=torch.float64, device=dev)
        chunk = max(1, st['cells'] // max(1, n))
        Nf, nk, g = st['Nf'], st['nk'], st['g']
        for a in range(0, m, chunk):
            vv = v[a:a + chunk]
            idx = (b[:, None] + vv[None, :]) % Ld + po[:, None]      # (positions x values) buckets
            Nc = Nf[idx * C + c].long()                              # N[bucket, class]
            cnt = nk[idx]                                            # bucket totals
            out_p[a:a + chunk] = (g[cnt] - g[Nc]).sum(0)             # exact marginal CE
            out_r[a:a + chunk] = (Nc == 0).sum(0)                    # new (bucket,class) cells
        return out_p, out_r

    def _score(self, base, pos_off, cls, vals):
        m = len(vals)
        w = np.array([d['share'] for d in self._dev], dtype=np.float64)
        cuts = np.concatenate([[0], np.round(np.cumsum(w) / w.sum() * m)]).astype(int)
        order = sorted(range(len(self._dev)), key=lambda i: self._dev[i]['dev'].type == 'cpu')
        res = {}
        with torch.inference_mode():
            for i in order:                    # GPUs first: their kernels run async while we go on
                lo, hi = cuts[i], cuts[i + 1]
                if hi > lo:
                    res[i] = self._score_on(self._dev[i], base, pos_off, cls, vals[lo:hi])
            P_out, R_out = np.empty(m), np.empty(m)
            for i, (p, r) in res.items():      # this .cpu() is the only sync point
                lo, hi = cuts[i], cuts[i + 1]
                P_out[lo:hi] = p.cpu().numpy()
                R_out[lo:hi] = r.cpu().numpy()
        return P_out, R_out

    def _move_parallel(self, P, cur, vals, ntry):
        P = np.asarray(P, dtype=np.int64)
        old = self.h[P].copy()
        d_rm, c_rm = self._apply(P, old, -1)

        base = (old - cur) % self.local_D
        pos_off = self.P_pos[P] * self.local_D
        primary_cost, reuse_cost = self._score(base, pos_off, self.P_c[P], vals)

        score = primary_cost + self.reuse_tiebreak * reuse_cost
        score = score + self.rng.random(len(vals)) * 1e-7

        for vi in np.argsort(score)[:ntry]:
            hn = (base + vals[vi]) % self.local_D + pos_off
            self.h[P] = hn
            d_in, c_in = self._apply(P, hn, +1)
            dp, dc = d_rm + d_in, c_rm + c_in
            if dp < -1e-9 or (abs(dp) <= 1e-9 and dc <= 0):
                return int(vals[vi])
            self._apply(P, hn, -1)
        self.h[P] = old
        self._apply(P, old, +1)
        return int(cur)

    def _move(self, P, cur, vals, ntry=8):
        if self.loss != 'ce' or not self._mirror_ready:
            return super()._move(P, cur, vals, ntry=ntry)   # original path (also ticks the bar)
        out = self._move_parallel(P, cur, vals, ntry)
        self._bar.update(1)
        return out


## Sparse scoring: `SparseGPUAlignedHashTrainer`

Same search, same acceptance rule, same scores as `GPUAlignedHashTrainer` - only the candidate scoring inside each coordinate move changes.

**Idea.** A candidate value shifts every affected address by the same amount (mod `local_D`), and an *empty* bucket costs exactly 0 in the CE loss. So instead of gathering a `positions x candidates` block from the `D x C` count table, walk the **occupied** buckets of each slot and scatter-add each one's cost onto the single candidate value that would land a position on it.

| | cost per move |
|---|---|
| dense (previous) | `O(positions x candidates)` random reads |
| sparse (this cell) | `O(positions x occupied buckets in the slot)`, independent of the number of candidates |

The biggest win is on `s[token]` parameters (`local_D` candidates each). `sparse_mode='auto'` (set in `CFG`) picks the cheaper path per move, so dense slots still use the GPU/CPU gather. Use `'always'` or `'never'` to force a path.

Notes: the sparse path runs in numpy on the CPU master copy of the count table. If you rely on it mostly, `devices=['cpu']` avoids the cost of syncing GPU mirrors. `verify_sparse_scoring(trainer)` checks sparse against dense on a few random parameters.

In [ ]:
# ---- SPARSE SCORING ---------------------------------------------------------
# Run AFTER the GPUAlignedHashTrainer cell. Same search, same acceptance rule, same
# scores -- only the candidate scoring inside each coordinate move changes.
#
# Why it is faster: every candidate value shifts all affected addresses by the same
# amount mod local_D, and an EMPTY bucket contributes exactly 0 to the CE cost
# (g[0] - g[0] = 0). So instead of gathering `positions x candidates` cells, we walk the
# OCCUPIED buckets of each slot and scatter-add each one's cost onto the one candidate
# value that would land a position on it:
#
#     dense : O(|P| x |vals|)                    random reads of the D x C table
#     sparse: O(|P| x occupied buckets in slot)  independent of |vals|
#
# sparse_mode = 'auto'   -> per move, use whichever is cheaper (default)
#               'always' -> always sparse      'never' -> always the parent's dense scoring
import numpy as np


class SparseScoreMixin:
    sparse_mode = 'auto'
    sparse_cells = 2 ** 22          # max (positions x occupied buckets) block held at once

    def _g_table(self):
        g = getattr(self, '_g_cache', None)
        if g is None or len(g) != self.NP + 1:
            g = self.flogf[1:self.NP + 2] - self.flogf[:self.NP + 1]
            self._g_cache = g
        return g

    def _score(self, base, pos_off, cls, vals):
        if self.sparse_mode == 'never':
            return super()._score(base, pos_off, cls, vals)

        Ld, C = self.local_D, self.C
        base = np.asarray(base, dtype=np.int64)
        cls = np.asarray(cls, dtype=np.int64)
        vals = np.asarray(vals, dtype=np.int64)
        pos = np.asarray(pos_off, dtype=np.int64) // Ld
        slots, inv = np.unique(pos, return_inverse=True)
        counts = np.bincount(inv)

        # occupied buckets per slot (local offsets). nk is the up-to-date numpy master copy.
        occ = [np.flatnonzero(self.nk[s * Ld:(s + 1) * Ld]) for s in slots]
        if self.sparse_mode == 'auto':
            sparse_cost = sum(int(counts[k]) * len(o) for k, o in enumerate(occ))
            if sparse_cost >= len(base) * len(vals):
                return super()._score(base, pos_off, cls, vals)

        g = self._g_table()
        prim = np.zeros(Ld)                 # primary cost, indexed by candidate residue
        hits = np.zeros(Ld)                 # positions landing on an existing (bucket, class) cell
        for k, s in enumerate(slots):
            loc = occ[k]
            if loc.size == 0:
                continue
            ob = loc + s * Ld               # global bucket ids
            nkg = g[self.nk[ob]]
            sel = np.flatnonzero(inv == k)
            step = max(1, self.sparse_cells // loc.size)
            for a in range(0, sel.size, step):
                ii = sel[a:a + step]
                Nc = self.Nf[ob[None, :] * C + cls[ii][:, None]].astype(np.int64)
                w = nkg[None, :] - g[Nc]                        # marginal CE of landing there
                v = (loc[None, :] - base[ii][:, None]) % Ld     # candidate that lands there
                prim += np.bincount(v.ravel(), weights=w.ravel(), minlength=Ld)
                hits += np.bincount(v[Nc > 0], minlength=Ld)

        r = vals % Ld
        return prim[r], len(base) - hits[r]             # (primary_cost, reuse_cost)


class SparseGPUAlignedHashTrainer(SparseScoreMixin, GPUAlignedHashTrainer):
    """GPUAlignedHashTrainer with sparse candidate scoring. Falls back to the dense
    GPU/CPU scoring on moves where that is cheaper (sparse_mode='auto')."""
    def __init__(self, *a, sparse_mode='auto', sparse_cells=2 ** 22, **kw):
        self.sparse_mode = sparse_mode
        self.sparse_cells = int(sparse_cells)
        super().__init__(*a, **kw)


def verify_sparse_scoring(trainer, n_params=6, atol=1e-6, seed=0):
    """Sanity check: on a few random parameters, sparse and dense scoring must agree."""
    rng = np.random.default_rng(seed)
    keys = [('s', p) for p in trainer.s_params] + [('d', p) for p in trainer.d_params]
    saved = trainer.sparse_mode
    for i in rng.choice(len(keys), size=min(n_params, len(keys)), replace=False):
        kind, p = keys[i]
        if kind == 's':
            P, cur, vals = trainer.occ_s[p], trainer.s[p], trainer.vals_s
        else:
            P, vals = trainer.occ_d[p], trainer.vals_d
            cur = trainer.d[p[0] + trainer.R, p[1]]
        old = trainer.h[P].copy()
        trainer._apply(P, old, -1)                      # score as _move does: P removed
        try:
            base = (old - cur) % trainer.local_D
            pos_off = trainer.P_pos[P] * trainer.local_D
            trainer.sparse_mode = 'always'
            p1, r1 = trainer._score(base, pos_off, trainer.P_c[P], vals)
            trainer.sparse_mode = 'never'
            p2, r2 = trainer._score(base, pos_off, trainer.P_c[P], vals)
        finally:
            trainer.sparse_mode = saved
            trainer._apply(P, old, +1)
        assert np.allclose(p1, p2, atol=atol), f"primary mismatch on {kind} {p}"
        assert np.array_equal(np.round(r1), np.round(r2)), f"reuse mismatch on {kind} {p}"
    print(f"sparse scoring matches dense on {min(n_params, len(keys))} random parameters")

## Train

The only architectural change is the hash: **still one lookup per ball, no layering**.

`train_with_restarts` now builds a `GPUAlignedHashTrainer`, so it uses all GPUs (or all CPU cores) automatically.


### Faster exact `trainer.compile()`

The physical-deflector compiler now uses deletion-only predecessor/successor indexes with path compression instead of scanning every distance for each free node. It preserves nearest-node selection, left-first distance ties, the sink, chain order, expansion sizes, hash parameters, and energy calculation. This optimization needs no additional dependency and retains the earlier splitting/resegmentation acceleration. It adds two temporary O(D) node indexes during compilation.

Validation: 1,000 randomized compiler states (including no free nodes, all free nodes, empty emissions, Unicode, and dynamic expansion) produced identical complete deflector tables, hash arrays, energy settings, diagnostic output, and emitted strings at every node. Fifty repeat compilations also matched. The full training dataset was not run.

Synthetic compilation benchmarks with precomputed table strings: 2,000 starting buckets at 50% occupancy, 0.203 s → 0.00517 s (~39×); 5,000 at 95% occupancy, 5.437 s → 0.0268 s (~203×). Emission lengths ranged from zero to eight characters. These are compiler allocation benchmarks, not training speedups; profiling the learned count table can add time in actual use.

Restart and run all cells to use the updated compiler. If retaining an already-trained trainer in the current kernel, rerun the HashTrainer definition cell and then execute `type(trainer).compile = HashTrainer.compile` before `brain = trainer.compile()`; training does not need to be repeated for this compiler-only update.


In [ ]:
import pickle
trainer = train_with_restarts(tiny_cases)
verify_sparse_scoring(trainer)   # sparse == dense scoring on a few random parameters (cheap)
brain = trainer.compile()

# Save the brain to a file
with open('brain.pkl', 'wb') as f:
    pickle.dump(brain, f)
print(len(brain.deflectors_list))

training: initial split:   0%|          | 0/20333 [00:00<?, ?ex/s]

[parallel] scoring on: cpu (share 1)  | count table 52000x4544 int16 = 0.47 GB per GPU


training:   0%|          | 0.00/57.3k [00:00<?, ?coord/s]

resegment:   0%|          | 0/20333 [00:00<?, ?ex/s]

restart 0: primary=56065.1968  conflicts=17634  train-table-exact=0.2596  reuse=565725
selected: primary=56065.1968  conflicts=17634  train-table-exact=0.2596
sparse scoring matches dense on 6 random parameters
compiled: start-buckets=31336  chain-nodes=2807  longest-emission=5  longest-hop=4641  ball-energy=4657
52000


## Evaluate on the real discrete simulator

In [ ]:
tr_exact, tr_ratio = evaluate(brain, tiny_cases)
te_exact, te_ratio = evaluate(brain, val_cases[:1000])
print(f"train: exact-match={tr_exact:.4f}  similarity={tr_ratio:.4f}")
print(f"test : exact-match={te_exact:.4f}  similarity={te_ratio:.4f}")

print("\nTraining examples:")
for inp, ans in tiny_cases[:3]:
    pred = brain.compute_input(inp)
    print(f"{inp!r:12} -> {pred!r:6}  target={ans!r}")

print("\nHeld-out examples:")
for inp, ans in sorted(val_cases)[:10]:
    pred = brain.compute_input(inp)
    print(f"{inp!r:12} -> {pred!r:6}  target={ans!r}")

train: exact-match=0.2596  similarity=0.3475
test : exact-match=0.0170  similarity=0.1575

Training examples:
'Once upon a time there was a little boy. He was three years old and loved to eat. One day, he saw a spicy snack. He' -> ''      target='wanted'
'upon a time there was a little boy. He was three years old and loved to eat. One day, he saw a spicy snack. He wanted' -> 'to'    target='to'
'a time there was a little boy. He was three years old and loved to eat. One day, he saw a spicy snack. He wanted to' -> ''      target='try'

Held-out examples:
'! Ow! Ow!"\n\nLily dropped her pipe and ran to the hole. She saw Tom lying on the ground, holding his foot.' -> 'athedollkemell'  target='He'
'! Ow!"\n\nLily dropped her pipe and ran to the hole. She saw Tom lying on the ground, holding his foot. He had' -> 'ownfaceye'  target='shot'
'!"\n\n"Help, help!" said Lily, hiding behind her shield. "The dragon is too strong!"\n\nMax jumped on Lily and' -> 'huggeSorryomeey'  target='l'
'!"\n\nB

In [ ]:

# for _ in range(10):
#   sentence = input("Input: ")
#   sentence = brain.compute_input(sentence)
#   print(sentence)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Extract deflectors details: [destination, value]
deflectors = brain.deflectors_list

# Separate into values: strings (emitted symbols) vs integers (bounce energy)
target_values = []
for dest, val in deflectors:
    if isinstance(val, int):
        if val < -1e6:
            target_values.append("Unused (Trap / -1B)")
        else:
            target_values.append(f"Int Bounce ({val})")
    else:
        target_values.append(f"Token ('{val}')")

df_targets = pd.DataFrame({'Target Category': target_values})
counts = df_targets['Target Category'].value_value_counts() if hasattr(df_targets['Target Category'], 'value_value_counts') else df_targets['Target Category'].value_counts()

# Plot distribution
plt.figure(figsize=(10, 5))
sns.barplot(x=counts.values, y=counts.index, palette='viridis')
plt.title('Distribution of Deflector Targets in the Deflector Table')
plt.xlabel('Count')
plt.ylabel('Target Category')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# We will simulate a walk for each possible starting node (0 to D-1) and record the step count.
def get_bounce_counts(brain):
    step_counts = []
    dl = brain.deflectors_list
    cap = brain.cap

    for start_node in range(brain.D):
        be = brain.ball_energy
        node = start_node
        steps = 0

        while be > 0 and steps < cap:
            dest, val = dl[node]
            dist = abs(dest - node)
            if isinstance(val, int):
                be += val - dist
            else:
                be += 1 - dist
            node = dest
            steps += 1

        step_counts.append(steps)
    return step_counts

bounce_counts = get_bounce_counts(brain)

# Plot the results
plt.figure(figsize=(12, 5))

# Left subplot: Histogram of step distributions
plt.subplot(1, 2, 1)
plt.hist(bounce_counts, bins=np.arange(0, brain.cap + 2) - 0.5, rwidth=0.8, color='skyblue', edgecolor='black')
plt.title('Distribution of Bounce Counts per Ball')
plt.xlabel('Number of Bounces (Steps)')
plt.ylabel('Number of Balls (Nodes)')
plt.xticks(range(0, brain.cap + 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Right subplot: Sequence plot across all D nodes
plt.subplot(1, 2, 2)
plt.plot(bounce_counts, color='teal', alpha=0.6, linewidth=0.5)
plt.title('Bounce Counts by Node Index')
plt.xlabel('Node Index')
plt.ylabel('Bounces')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Notes and limits

* **What changed.** The original global hash was an associative-memory address. The new
  address is a reusable local feature hash: the same `(relative position, token)` feature
  is shared across all examples, and each ball position has its own bucket region.
* **Why this can generalize.** A test context does not need to reproduce an entire training
  context. It only needs the relevant local features to land in buckets whose table entries
  were learned from other examples.
* **What did not change.** There is still exactly one lookup per ball. The walk never reads
  another input token after hashing, so this is not the proposed layered architecture.
* **Training objective.** Class purity / profiled CE remains primary. Reuse is only a
  tie-breaker, which prevents the optimizer from sacrificing fit merely to get more sharing.
* **Capacity knob.** `local_D` controls buckets per slot. Smaller values put more pressure
  on feature reuse; larger values make the search closer to memorization.
* **Locality knob.** `hash_radius` controls how much context a ball can use. Increasing it
  raises expressive power but weakens the inductive bias toward reuse.
